In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata
from datetime import date

In [2]:
df = pd.read_csv("output_woche4.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47803 entries, 0 to 47802
Columns: 147 entries, Unnamed: 0 to RemoteCategoryNum
dtypes: float64(41), int64(2), object(104)
memory usage: 53.6+ MB


In [3]:
thresh_rows = 90  # mindestens 90 ausgefüllte Spalten pro Zeile
df_rows = df.dropna(axis=0, thresh=thresh_rows)

print("vorher:", df.shape)
print("nachher:", df_rows.shape)
df_rows.info()
df_rows.to_csv("survey_results_public_reduced.csv", index=False)
print("Saved:", df_rows.shape, "->", "survey_results_public_reduced.csv")

vorher: (47803, 147)
nachher: (22154, 147)
<class 'pandas.core.frame.DataFrame'>
Index: 22154 entries, 0 to 47801
Columns: 147 entries, Unnamed: 0 to RemoteCategoryNum
dtypes: float64(41), int64(2), object(104)
memory usage: 25.0+ MB
Saved: (22154, 147) -> survey_results_public_reduced.csv


In [4]:
IN_PATH = "survey_results_public_reduced.csv"
OUT_PATH = "survey_results_cleaned.csv"

df = pd.read_csv(IN_PATH)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22154 entries, 0 to 22153
Columns: 147 entries, Unnamed: 0 to RemoteCategoryNum
dtypes: float64(41), int64(2), object(104)
memory usage: 24.8+ MB


In [5]:
drop_cols = [
    'EmploymentAddl', 'LearnCodeChoose', 'LearnCode', 'AILearnHow', 'PurchaseInfluence',
    'ToolCountWork', 'ToolCountPersonal',
    'LanguageAdmired', 'LanguagesHaveEntry', 'LanguagesWantEntry',
    'DatabaseAdmired', 'DatabaseHaveEntry', 'DatabaseWantEntry',
    'PlatformAdmired', 'PlatformHaveEntry', 'PlatformWantEntry',
    'WebframeAdmired', 'WebframeHaveEntry', 'WebframeWantEntry',
    'DevEnvsAdmired', 'DevEnvHaveEntry', 'DevEnvWantEntry',
    'OpSysPersonal use', 'OpSysProfessional use',
    'OfficeStackAsyncAdmired', 'OfficeStackHaveEntry', 'OfficeStackWantEntry',
    'CommPlatformAdmired', 'CommPlatformHaveEntr', 'CommPlatformWantEntr',
    'AIModelsAdmired', 'AIModelsHaveEntry', 'AIModelsWantEntry',
    'AISent', 'AIAcc', 'AIComplex',
    'AIToolCurrently partially AI', "AIToolDon't plan to use AI for this task",
    'AIToolPlan to partially use AI', 'AIToolPlan to mostly use AI',
    'AIToolCurrently mostly AI', 'AIFrustration', 'AIExplain',
    'AIAgentChange', 'AgentUsesGeneral',
    'AIAgentImpactSomewhat agree', 'AIAgentImpactNeutral',
    'AIAgentImpactSomewhat disagree', 'AIAgentImpactStrongly agree',
    'AIAgentImpactStrongly disagree',
    'AIAgentChallengesNeutral', 'AIAgentChallengesSomewhat disagree',
    'AIAgentChallengesStrongly agree', 'AIAgentChallengesSomewhat agree',
    'AIAgentChallengesStrongly disagree',
    'AIAgentKnowledge', 'AIAgentKnowWrite',
    'AIAgentOrchestration', 'AIAgentOrchWrite',
    'AIAgentObserveSecure', 'AIAgentObsWrite',
    'AIAgentExternal', 'AIAgentExtWrite',
    'AIHuman', 'AIOpen', 'LanguageWantToWorkWith', 'DatabaseWantToWorkWith', 'PlatformWantToWorkWith', 'WebframeWantToWorkWith', 'DevEnvsWantToWorkWith', 'OfficeStackAsyncWantToWorkWith', 'AIModelsWantToWorkWith', 'CommPlatformWantToWorkWith'
]
prefixes_drop = ("TechEndorse", "TechOppose", "JobSatPoints", "SO")

# age_map = {
#     "Under 18 years old": 17,
#     "18-24 years old": 21,
#     "25-34 years old": 29,
#     "35-44 years old": 39,
#     "45-54 years old": 49,
#     "55-64 years old": 59,
#     "65 years or older": 70
# }
# age_map2 = {
#     "Under 18 years old": 18,
#     "18-24 years old": 24,
#     "25-34 years old": 34,
#     "35-44 years old": 44,
#     "45-54 years old": 54,
#     "55-64 years old": 64,
#     "65 years or older": 100
# }

multi_select_cols = [
    'LanguageHaveWorkedWith',
    'DatabaseHaveWorkedWith',
    'PlatformHaveWorkedWith',
    'WebframeHaveWorkedWith',
    'DevEnvsHaveWorkedWith',
    'OfficeStackAsyncHaveWorkedWith',
    'AIModelsHaveWorkedWith',
    'AIAgent_Uses'
]

In [6]:
def insert_mapped_column(df, base_col, new_col, mapping):
    if base_col not in df.columns:
        return
    df.insert(df.columns.get_loc(base_col) + 1, new_col, df[base_col].map(mapping))

def clean_text(s):
    if pd.isna(s):
        return ""
    s = str(s).strip()
    s = unicodedata.normalize("NFC", s)
    s = s.replace("–", "-").replace("—", "-").replace("’", "'")
    s = re.sub(r"\s+", " ", s)
    return s

def to_lowercase(s):
    if pd.isna(s):
        return s
    return str(s).lower()

def clean_multi_select_to_str(value):
    """'A;B;C' -> 'a;b;c' (unique+sorted). NaN -> '' """
    if pd.isna(value):
        return ""
    parts = [clean_text(p).lower() for p in str(value).split(";")]
    parts = sorted(set([p for p in parts if p]))
    return ";".join(parts)

def parse_years(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s == "":
        return np.nan
    try:
        return float(s)
    except:
        return np.nan


In [7]:
df = pd.read_csv(IN_PATH)

# Drop unneeded columns (robust)
df = df.drop(columns=drop_cols, errors="ignore")
df = df.drop(columns=df.columns[df.columns.str.startswith(prefixes_drop)], errors="ignore")

# Remote mapping + missing flag + fill (0.5)
# insert_mapped_column(df, "RemoteWork", "RemoteCategoryNum", remote_map)
# if "RemoteCategoryNum" in df.columns:
#     df["RemoteMissing"] = df["RemoteCategoryNum"].isna().astype("int8")
#     df["RemoteCategoryNum"] = df["RemoteCategoryNum"].fillna(0.5)

# Age mapping + filter <=65
# insert_mapped_column(df, "Age", "AgeNum", age_map)
# # insert_mapped_column(df, "Age", "MaxAge", age_map2) -> brauchen wir glaub nicht oder
# if "AgeNum" in df.columns:
#     df = df[df["AgeNum"].notna() & (df["AgeNum"] <= 65)].copy()

# YearsCode / WorkExp numeric
if "YearsCode" in df.columns:
    df["YearsCode"] = df["YearsCode"].apply(parse_years)
if "WorkExp" in df.columns:
    df["WorkExp"] = pd.to_numeric(df["WorkExp"], errors="coerce")

# Lowercase object columns except Country/Currency
exclude_obj = {"Country"}
for col in df.select_dtypes(include=["object"]).columns:
    if col not in exclude_obj:
        df[col] = df[col].apply(to_lowercase)

# Multi-select cleanup (keep as string with ;)
for col in [c for c in multi_select_cols if c in df.columns]:
    df[col] = df[col].apply(clean_multi_select_to_str)

# Plausibility filters
if {"WorkExp", "AgeNum"}.issubset(df.columns):
    df = df[~(df["WorkExp"] > (df["AgeNum"] - 16))].copy()
if {"YearsCode", "AgeNum"}.issubset(df.columns):
    df = df[~(df["YearsCode"] > (df["AgeNum"] - 6))].copy()

# # Salary conversion
# if "Currency" in df.columns:
#     df["Currency"] = df["Currency"].astype(str).str[:3]
#     df.loc[df["Currency"].isin(["nan", "none", "None"]), "Currency"] = np.nan
# if "CompTotal" in df.columns:
#     df["CompTotal"] = pd.to_numeric(df["CompTotal"], errors="coerce")
#
# if {"Currency", "CompTotal"}.issubset(df.columns):
#     df["ConvertedCompTotal"] = df.apply(lambda r: convert_to_usd(r["Currency"], r["CompTotal"]), axis=1)

# Drop raw salary cols AFTER conversion
# df = df.drop(columns=[c for c in ["CompTotal", "Currency", "ConvertedCompYearly"] if c in df.columns], errors="ignore")

# 95%-Filter (row removal) for numeric outliers
for col in [c for c in ["ConvertedCompTotal", "WorkExp", "YearsCode"] if c in df.columns]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    q98 = df[col].quantile(0.98)
    before = len(df)
    df = df[df[col].isna() | (df[col] <= q98)].copy()
    print(f"{col}: kept <= q98={q98:.4g} | removed {before - len(df)} rows")

# Save
df.to_csv(OUT_PATH, index=False)
print("Saved:", df.shape, "->", OUT_PATH)


ConvertedCompTotal: kept <= q98=3.245e+05 | removed 314 rows
WorkExp: kept <= q98=35 | removed 386 rows
YearsCode: kept <= q98=40 | removed 336 rows
Saved: (19965, 38) -> survey_results_cleaned.csv


In [8]:
IN_PATH = "survey_results_cleaned.csv"
OUT_PATH = "survey_results_cleaned_final.csv"

df = pd.read_csv(IN_PATH)

# df.info()
thresh_rows = 32  # z.B. mindestens 32 ausgefüllte Spalten pro Zeile
df_rows = df.dropna(axis=0, thresh=thresh_rows)

print("vorher:", df.shape)
print("nachher:", df_rows.shape)
df_rows.info()
df_rows.to_csv(OUT_PATH, index=False)
print("Saved:", df_rows.shape, "->", OUT_PATH)

vorher: (19965, 38)
nachher: (18603, 38)
<class 'pandas.core.frame.DataFrame'>
Index: 18603 entries, 0 to 19964
Data columns (total 38 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Unnamed: 0                      18603 non-null  int64  
 1   ResponseId                      18603 non-null  int64  
 2   MainBranch                      18603 non-null  object 
 3   Age                             18603 non-null  object 
 4   EdLevel                         18592 non-null  object 
 5   Employment                      18603 non-null  object 
 6   Country                         18603 non-null  object 
 7   WorkExp                         18276 non-null  float64
 8   LearnCodeAI                     18589 non-null  object 
 9   YearsCode                       18554 non-null  float64
 10  DevType                         18603 non-null  object 
 11  OrgSize                         17326 non-null  object 
 

In [9]:
df_rows.ConvertedCompTotal.describe()

count     14290.000000
mean      90267.489323
std       61123.720567
min           0.000000
25%       46712.000000
50%       80000.000000
75%      120876.365727
max      323000.000000
Name: ConvertedCompTotal, dtype: float64